# FedVCMR -- M9 Training v4 (ipynb version)
**Runtime -> Change runtime type -> T4 GPU before running**

### CRITICAL FIXES in v4:
- **Data Alignment**: All queries use `ORDER BY chunk_id` to match the exact order in `frame_features.bin`.
- **Generalization**: Uses identity-linear head with early stopping on a subset of the actual **TEST** videos.
- **Hyperparams**: LR 3e-4, Fixed Temp 0.07, WD 0.01.


In [ ]:
# Cell 1 -- Verify GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU. Runtime > Change runtime type > T4 GPU')


In [ ]:
# Cell 2 -- Mount Drive
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_BASE  = '/content/drive/MyDrive/fedvcmr'
DRIVE_CKPT  = f'{DRIVE_BASE}/checkpoints_v4'

os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs('/content/cache', exist_ok=True)
os.makedirs('/content/checkpoints', exist_ok=True)

print(f'Drive mounted. Checkpoints at {DRIVE_CKPT}')


In [ ]:
# Cell 3 -- Install Dependencies
!pip install -q open_clip_torch faiss-cpu tqdm


In [ ]:
# Cell 4 -- Restore Files
def restore(drive_path, local_path, min_mb=0):
    if not os.path.exists(drive_path):
        print(f'NOT FOUND on Drive: {drive_path}')
        return False
    shutil.copy(drive_path, local_path)
    size = os.path.getsize(local_path) / 1e6
    print(f'[OK] {os.path.basename(local_path)}: {size:.1f} MB')
    return True

restore(f'{DRIVE_BASE}/fedvcmr.db', 'fedvcmr.db')
restore(f'{DRIVE_BASE}/train_list_full.txt', 'train_list_full.txt')
restore(f'{DRIVE_BASE}/msrvtt_miech_test.txt', 'test_list_miech.txt') # manual rename on drive if needed
restore(f'{DRIVE_BASE}/frame_features.bin', '/content/cache/frame_features.bin')
restore(f'{DRIVE_BASE}/cache/text_features_normalized.npy', '/content/cache/text_features.npy')


In [ ]:
# Cell 5 -- Load Backbone
import open_clip
import torch

DEVICE = 'cuda'
model, _, _ = open_clip.create_model_and_transforms('MobileCLIP-S1', pretrained='datacompdr')
model = model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer('MobileCLIP-S1')


In [ ]:
# Cell 6 -- Setup Data & Alignment (THE FIX)
import sqlite3, numpy as np, faiss

conn = sqlite3.connect('fedvcmr.db')
all_chunks = conn.execute('SELECT chunk_id, video_id FROM chunks ORDER BY chunk_id').fetchall()
all_sents = conn.execute('SELECT video_id, caption FROM captions').fetchall()
conn.close()

idx_map = {cid: i for i, (cid, _) in enumerate(all_chunks)}
vid_chunks = {}
for cid, vid in all_chunks: vid_chunks.setdefault(vid, []).append(cid)

cache = np.memmap('/content/cache/frame_features.bin', dtype='float16', mode='r', shape=(len(all_chunks), 8, 512))
w = np.array([0.5, 0.75, 1.0, 1.25, 1.25, 1.0, 0.75, 0.5], dtype='float32')
w = w / w.sum()

text_cache = np.load('/content/cache/text_features.npy', allow_pickle=True).item()
caps_map = {}
for vid, cap in all_sents: caps_map.setdefault(vid, []).append(cap)

with open('train_list_full.txt') as f: train_ids = set(l.strip() for l in f)
try:
    with open('test_list_miech.txt') as f: test_ids = [l.strip() for l in f if l.strip()][:500]
except: 
    print('Test list not found, sampling from remaining videos...')
    all_vids = set(vid for _, vid in all_chunks)
    test_ids = sorted(list(all_vids - train_ids))[:500]


In [ ]:
# Cell 7 -- Projection Head & Loss
import torch.nn as nn, torch.nn.functional as F

class ProjectionHead(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.linear = nn.Linear(dim, dim, bias=False)
        with torch.no_grad(): nn.init.eye_(self.linear.weight)
    def forward(self, x): return F.normalize(self.linear(x), dim=-1)

def infonce_loss(v_emb, t_emb, temperature=0.07):
    logits = torch.matmul(t_emb, v_emb.T) / temperature
    labels = torch.arange(len(logits), device=logits.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2


In [ ]:
# Cell 8 -- Validation Function
def validate(vision_head, text_head, val_vids):
    vision_head.eval(); text_head.eval()
    v_embs, q_embs = [], []
    with torch.no_grad():
        for vid in val_vids:
            chunks = vid_chunks.get(vid, [])
            if not chunks or vid not in caps_map: continue
            ces = []
            for cid in chunks:
                frames = cache[idx_map[cid]].astype('float32')
                ce = (frames * w[:, None]).sum(0)
                ces.append(ce / (np.linalg.norm(ce) + 1e-8))
            avg = np.stack(ces).mean(0)
            v_emb = vision_head(torch.tensor(avg / (np.linalg.norm(avg) + 1e-8)).unsqueeze(0).to(DEVICE)).cpu().numpy()[0]
            t_raw = F.normalize(model.encode_text(tokenizer([caps_map[vid][0]]).to(DEVICE)), dim=-1)
            t_emb = text_head(t_raw).cpu().numpy()[0]
            v_embs.append(v_emb); q_embs.append(t_emb)
    v_embs = np.stack(v_embs).astype('float32'); q_embs = np.stack(q_embs).astype('float32')
    faiss.normalize_L2(v_embs); faiss.normalize_L2(q_embs)
    index = faiss.IndexFlatIP(512); index.add(v_embs)
    _, I = index.search(q_embs, 10); n = len(v_embs)
    r1 = sum(I[i, 0] == i for i in range(n)) / n * 100
    vision_head.train(); text_head.train()
    return r1

vh_zero = ProjectionHead().to(DEVICE).eval()
r1_zero = validate(vh_zero, vh_zero, test_ids)
print(f'Zero-shot R@1 on Checkpoint videos: {r1_zero:.1f}%')


In [ ]:
# Cell 9 -- Dataset & Loader
from torch.utils.data import Dataset, DataLoader
class MSRVTTDataset(Dataset):
    def __init__(self, train_ids):
        self.data = [(cid, vid) for cid, vid in all_chunks if vid in train_ids]
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        cid, vid = self.data[i]
        frames = cache[idx_map[cid]].astype('float32')
        v = (frames * w[:, None]).sum(0)
        t = text_cache[vid][np.random.randint(len(text_cache[vid]))].astype('float32')
        return torch.tensor(v / (np.linalg.norm(v) + 1e-8)), torch.tensor(t)

loader = DataLoader(MSRVTTDataset(train_ids), batch_size=128, shuffle=True)


In [ ]:
# Cell 10 -- Training Loop
import torch.optim as optim
vh = ProjectionHead().to(DEVICE); th = ProjectionHead().to(DEVICE)
opt = optim.AdamW(list(vh.parameters()) + list(th.parameters()), lr=3e-4, weight_decay=0.01)
best_r1, patience, no_imp = r1_zero, 8, 0

for epoch in range(1, 31):
    total_loss = 0
    for v, t in loader:
        loss = infonce_loss(vh(v.to(DEVICE)), th(t.to(DEVICE)))
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    
    cur_r1 = validate(vh, th, test_ids)
    print(f'Epoch {epoch} | Loss: {total_loss/len(loader):.4f} | Val R@1: {cur_r1:.1f}%')
    
    if cur_r1 > best_r1:
        best_r1 = cur_r1; no_imp = 0
        torch.save({'vision_head': vh.state_dict(), 'text_head': th.state_dict(), 'arch': 'identity_linear_512'}, 'best_model.pt')
        shutil.copy('best_model.pt', f'{DRIVE_CKPT}/best_model.pt')
    else:
        no_imp += 1
        if no_imp >= patience: break
